# Sections 6 & 7: Machine Learning — Predictive Modeling & Demographic Clustering

## Overview
This notebook applies supervised machine learning and unsupervised clustering to the curated exoplanet dataset (`cleaned_exoplanet_data.csv`).

### Section 6: Machine Learning — Planetary Property Prediction
* **Task**: Predict planetary mass ($\log_{10} M_p [M_\oplus]$) from observed orbital parameters, planetary radius, and host-star properties.
* **Astrophysical Motivation**: Transit surveys (Kepler, TESS) directly measure planet radius $R_p$, but obtaining dynamical mass requires resource-intensive ground-based Radial Velocity (RV) campaigns or Transit Timing Variations (TTV). Reliable ML prediction of planetary mass is therefore a major accelerator for atmospheric characterization (e.g. with JWST).
* **Scientific Integrity**: To avoid circular data leakage, models are trained and validated **strictly on empirical dynamical mass measurements** (`is_measured_mass == True`), excluding points derived from theoretical Mass-Radius relations. Preprocessing (StandardScaler) is wrapped inside sklearn **Pipelines** so scaling statistics are computed only on training folds and never leak from validation data.
* **Models Evaluated**:
  1. Ridge Regression (L2 regularized linear model)
  2. Lasso Regression (L1 regularized sparse linear model)
  3. $k$-Nearest Neighbors Regressor ($k$-NN)
  4. Support Vector Regressor (SVR with RBF kernel)
  5. Random Forest Regressor (Ensemble bagging)
  6. HistGradientBoosting Regressor (Ensemble gradient boosted trees)
* **Evaluation Metrics**: 5-Fold Cross-Validation $R^2$, Test $R^2$, Root Mean Squared Error (RMSE), Mean Absolute Error (MAE), and Median Absolute Error (MedAE).
* **Interpretability**: Permutation feature importance, residual analysis, and physical error attribution.

---

### Section 7: Machine Learning — Demographic Clustering
* **Task**: Discover natural, unsupervised groupings of exoplanets in physical parameter space.
* **Algorithms**: **$K$-Means** and **Gaussian Mixture Models (GMM)**.
* **Optimization**: Hyperparameter selection via the **Elbow Method (Inertia)**, **Silhouette Scores**, and **Bayesian Information Criterion (BIC)**.
* **Honest Assessment**: We run clustering on BOTH the measured-mass subsample (empirical_df) AND the full dataset (df) to understand selection effects.


In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split, KFold, GroupKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge, Lasso
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error, median_absolute_error
from sklearn.inspection import permutation_importance
from sklearn.cluster import KMeans
from sklearn.mixture import GaussianMixture
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.sans-serif'] = 'Arial'
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['figure.dpi'] = 120

os.makedirs('figures', exist_ok=True)
print("ML libraries imported successfully!")


---
# Section 6: Machine Learning — Prediction

### 6.1 Dataset Preparation & Leakage Prevention
We select exoplanets with **empirically measured masses** (`is_measured_mass == True`). We use log-scaled features to capture power-law astrophysical relationships:
* $\log_{10}(R_p)$ — Planet Radius [$R_\oplus$]
* $\log_{10}(P)$ — Orbital Period [days]
* $\log_{10}(a)$ — Semi-Major Axis [AU]
* $\log_{10}(S)$ — Insolation Flux [$S_\oplus$]
* $\log_{10}(T_{\text{eff}})$ — Stellar Effective Temperature [K]
* $\log_{10}(M_*)$ — Stellar Mass [$M_\odot$]
* $[\text{Fe/H}]$ — Stellar Metallicity [dex]
* $N_{\text{pl}}$ — Number of Planets in System
* $\log_{10}(d)$ — Distance to System [pc]


In [ ]:
df = pd.read_csv('cleaned_exoplanet_data.csv')

# Filter to empirical mass measurements only
empirical_df = df[df['is_measured_mass']].copy()

features = [
    'log_pl_rade', 'log_pl_orbper', 'log_pl_orbsmax', 'log_pl_insol',
    'log_st_teff', 'log_st_mass', 'st_met', 'sy_pnum', 'log_sy_dist'
]
target = 'log_pl_bmasse'

reg_df = empirical_df.dropna(subset=[target] + features).copy()
print(f"Sample size with complete empirical features: {len(reg_df):,} planets")

X = reg_df[features]
y = reg_df[target]

# 80/20 Train-Test Split with fixed seed (no pre-scaling — pipelines handle it)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)
print(f"Training set: {X_train.shape[0]} samples | Test set: {X_test.shape[0]} samples")

# --- FIX 6: Msini mass provenance note ---
print()
print("NOTE on mass provenance:")
print("  The is_measured_mass flag includes both Mass (true dynamical mass) and Msini")
print("  (minimum mass; true mass scaled by sin(i) where i is orbital inclination).")
print("  For RV planets without transit confirmation, mass is a lower bound with")
print("  mean statistical correction <sin(i)> = pi/4 ≈ 0.785.")


### 6.2 Model Training & Performance Comparison (Pipeline-Based, Leakage-Free)
We train six distinct regression architectures wrapped in sklearn **Pipelines** that include `StandardScaler` as the first step. This ensures that scaling statistics are **never computed on validation data**, eliminating the preprocessing data-leakage present in naive cross-validation approaches.

1. **Ridge Regression**: L2 regularization preventing multicollinearity between orbital and stellar features.
2. **Lasso Regression**: L1 regularization providing automated feature selection.
3. **$k$-Nearest Neighbors (k-NN)**: Non-parametric local similarity regression.
4. **Support Vector Regression (SVR)**: Non-linear kernel regression using Radial Basis Functions (RBF).
5. **Random Forest**: Non-linear ensemble of 150 decorrelated decision trees.
6. **HistGradientBoosting**: State-of-the-art gradient boosted decision trees.


In [ ]:
# FIX 1: All models wrapped in sklearn Pipelines — StandardScaler inside each pipeline
# X_train and X_test are passed RAW (unscaled); the pipeline handles scaling internally
# This eliminates preprocessing data leakage in cross-validation.

pipelines = {
    'Ridge': Pipeline([('scaler', StandardScaler()), ('model', Ridge(alpha=1.0))]),
    'Lasso': Pipeline([('scaler', StandardScaler()), ('model', Lasso(alpha=0.01))]),
    'kNN':   Pipeline([('scaler', StandardScaler()), ('model', KNeighborsRegressor(n_neighbors=7))]),
    'SVR (RBF)': Pipeline([('scaler', StandardScaler()), ('model', SVR(C=10.0, epsilon=0.1))]),
    'Random Forest': Pipeline([('scaler', StandardScaler()), ('model', RandomForestRegressor(n_estimators=150, max_depth=15, random_state=42, n_jobs=-1))]),
    'HistGradientBoosting': Pipeline([('scaler', StandardScaler()), ('model', HistGradientBoostingRegressor(max_iter=150, max_depth=8, random_state=42))])
}

benchmark_results = []
fitted_pipelines = {}

kf = KFold(n_splits=5, shuffle=True, random_state=42)

for name, pipe in pipelines.items():
    # Cross-validation on RAW X_train — pipeline scales internally per fold
    cv_r2 = cross_val_score(pipe, X_train, y_train, cv=kf, scoring='r2')

    # Fit on full training set
    pipe.fit(X_train, y_train)
    fitted_pipelines[name] = pipe

    # Predict on RAW X_test — pipeline applies the fitted scaler
    y_pred = pipe.predict(X_test)

    test_r2   = r2_score(y_test, y_pred)
    test_rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    test_mae  = mean_absolute_error(y_test, y_pred)
    test_medae = median_absolute_error(y_test, y_pred)

    benchmark_results.append({
        'Model': name,
        'CV R2 (Mean)': cv_r2.mean(),
        'CV R2 (Std)':  cv_r2.std(),
        'Test R2':      test_r2,
        'Test RMSE (dex)': test_rmse,
        'Test MAE (dex)':  test_mae,
        'Test MedAE (dex)': test_medae
    })
    print(f"{name:25s}  CV R2={cv_r2.mean():.4f}±{cv_r2.std():.4f}  Test R2={test_r2:.4f}  RMSE={test_rmse:.4f}  MedAE={test_medae:.4f}")

results_table = pd.DataFrame(benchmark_results).round(4)
print()
print("Regression Benchmark Performance Comparison (Pipeline-corrected):")
print(results_table.to_string(index=False))

# --- FIX 5: Log-scale error clarification ---
rf_row = results_table[results_table['Model'] == 'Random Forest'].iloc[0]
rf_rmse = rf_row['Test RMSE (dex)']
rf_medae = rf_row['Test MedAE (dex)']
print()
print("=== FIX 5: Log-scale error interpretation ===")
print(f"Errors are in log10(M_earth) units (dex).")
print(f"Random Forest RMSE = {rf_rmse:.3f} dex  -> multiplicative uncertainty = 10^{rf_rmse:.3f} = {10**rf_rmse:.2f}x")
print(f"Random Forest MedAE = {rf_medae:.3f} dex -> typical median multiplicative error = 10^{rf_medae:.3f} = {10**rf_medae:.2f}x ({(10**rf_medae - 1)*100:.0f}% relative uncertainty)")
print()
print("A RMSE of ~0.35 dex corresponds to a factor of 10^0.35 ≈ 2.24 multiplicative uncertainty in predicted mass.")
print("A MedAE of ~0.11 dex corresponds to a typical median multiplicative error of 10^0.11 ≈ 1.29 (29% typical relative uncertainty in mass).")


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 12))

# 1. R2 Score Comparison (CV vs Test)
ax = axes[0, 0]
x_pos = np.arange(len(results_table))
width = 0.35
ax.bar(x_pos - width/2, results_table['CV R2 (Mean)'], width, yerr=results_table['CV R2 (Std)'],
       capsize=4, label='5-Fold CV R2', color='#41b6c4', edgecolor='black', alpha=0.85)
ax.bar(x_pos + width/2, results_table['Test R2'], width, label='Test Set R2', color='#253494', edgecolor='black', alpha=0.85)
ax.set_xticks(x_pos)
ax.set_xticklabels(results_table['Model'], rotation=25, fontsize=10)
ax.set_ylim(0.65, 0.95)
ax.set_title('A. Model Explanatory Power (R2 Score) — Pipeline-corrected', fontsize=12, fontweight='bold')
ax.set_ylabel('R2 Score', fontsize=11)
ax.legend(frameon=True)

# 2. Error Metrics Comparison (RMSE & MAE)
ax = axes[0, 1]
ax.bar(x_pos - width/2, results_table['Test RMSE (dex)'], width, label='Test RMSE [dex]', color='#e6550d', edgecolor='black', alpha=0.85)
ax.bar(x_pos + width/2, results_table['Test MAE (dex)'], width, label='Test MAE [dex]', color='#fdae6b', edgecolor='black', alpha=0.85)
ax.set_xticks(x_pos)
ax.set_xticklabels(results_table['Model'], rotation=25, fontsize=10)
ax.set_title('B. Prediction Error Metrics (Lower is Better)', fontsize=12, fontweight='bold')
ax.set_ylabel('Error [log10 dex]', fontsize=11)
ax.legend(frameon=True)

# 3. Actual vs Predicted Mass (Random Forest pipeline)
ax = axes[1, 0]
best_pipe = fitted_pipelines['Random Forest']
best_preds = best_pipe.predict(X_test)
rf_idx = results_table[results_table['Model'] == 'Random Forest'].index[0]
scatter = ax.scatter(y_test, best_preds, c=X_test['log_pl_rade'], cmap='viridis', alpha=0.7, edgecolors='black', linewidth=0.4, s=32)
cbar = plt.colorbar(scatter, ax=ax)
cbar.set_label('log10(Rp [Rearth])', fontsize=10)
min_val = min(y_test.min(), best_preds.min()) - 0.2
max_val = max(y_test.max(), best_preds.max()) + 0.2
ax.plot([min_val, max_val], [min_val, max_val], 'r--', linewidth=2, label='1:1 Parity Line')
ax.set_xlim(min_val, max_val)
ax.set_ylim(min_val, max_val)
ax.set_title(f'C. Random Forest: Actual vs. Predicted Mass (R2={results_table.loc[rf_idx, "Test R2"]:.3f})', fontsize=12, fontweight='bold')
ax.set_xlabel('Actual log10(Mp [Mearth])', fontsize=11)
ax.set_ylabel('Predicted log10(Mp [Mearth])', fontsize=11)
ax.legend(frameon=True, loc='upper left')

# 4. Residual Distribution
ax = axes[1, 1]
residuals = y_test - best_preds
sns.histplot(residuals, bins=35, kde=True, color='#756bb1', edgecolor='black', ax=ax, alpha=0.6)
ax.axvline(0, color='red', linestyle='--', linewidth=1.5, label='Zero Bias')
ax.axvline(residuals.mean(), color='navy', linestyle=':', linewidth=1.5, label=f'Mean Error = {residuals.mean():.3f}')
ax.set_title(f'D. Residual Distribution (Std = {residuals.std():.3f} dex)', fontsize=12, fontweight='bold')
ax.set_xlabel('Residuals [log10(M_actual) - log10(M_pred)]', fontsize=11)
ax.set_ylabel('Test Sample Count', fontsize=11)
ax.legend(frameon=True)

plt.tight_layout()
plt.savefig('figures/06_ml_model_comparison.png', dpi=300)
plt.show()


### 6.3 Feature Importance & Astrophysical Interpretation
We examine what physical drivers the tree ensemble relies upon most to predict planet mass:
1. **MDI Importance (Mean Decrease in Impurity — MSE Reduction)**: Internal variance reduction in the regression decision trees. *Note: Gini impurity applies only to classification trees. For regression trees, MDI quantifies Mean Squared Error (MSE) reduction / variance explained per split. MDI importance reflects predictive reliance, not causation.*
2. **Permutation Importance**: Reduction in test score when the feature values are randomly shuffled (robust against cardinality bias).


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Extract Random Forest from its pipeline
rf_model = fitted_pipelines['Random Forest'].named_steps['model']

# 1. Random Forest MDI Feature Importance
# FIX 2: x-axis label corrected from "Relative Importance (Gini)" to MDI/MSE Reduction
ax = axes[0]
importances = rf_model.feature_importances_
indices = np.argsort(importances)
clean_labels = [f.replace('log_', '').replace('_', ' ').title() for f in np.array(features)[indices]]
ax.barh(range(len(indices)), importances[indices], color='#2c7fb8', edgecolor='black', alpha=0.85)
ax.set_yticks(range(len(indices)))
ax.set_yticklabels(clean_labels, fontsize=10)
ax.set_title('A. Random Forest: MDI Feature Importance\n(Regression Trees — MSE Reduction)', fontsize=12, fontweight='bold')
# FIX 2: Correct axis label — NOT Gini (classification); MSE Reduction (regression)
ax.set_xlabel('Mean Decrease in Impurity (MSE Reduction)', fontsize=11)
ax.text(0.97, 0.03, 'MDI reflects predictive reliance, not causation.',
        transform=ax.transAxes, fontsize=8, ha='right', va='bottom',
        style='italic', color='gray')

# 2. Permutation Importance — pass RAW X_test (pipeline handles scaling internally)
ax = axes[1]
perm = permutation_importance(fitted_pipelines['Random Forest'], X_test, y_test,
                               n_repeats=20, random_state=42)
perm_sorted = perm.importances_mean.argsort()
perm_labels = [f.replace('log_', '').replace('_', ' ').title() for f in np.array(features)[perm_sorted]]
ax.barh(range(len(perm_sorted)), perm.importances_mean[perm_sorted],
        xerr=perm.importances_std[perm_sorted],
        color='#41b6c4', edgecolor='black', alpha=0.85, capsize=4)
ax.set_yticks(range(len(perm_sorted)))
ax.set_yticklabels(perm_labels, fontsize=10)
ax.set_title('B. Permutation Importance (Test Set)\n(Pipeline-corrected; raw X_test passed)', fontsize=12, fontweight='bold')
ax.set_xlabel('Mean R2 Drop Upon Shuffling', fontsize=11)

plt.tight_layout()
plt.savefig('figures/06_ml_feature_importance.png', dpi=300)
plt.show()

print("Astronomical Interpretation:")
top_feat = np.array(features)[np.argmax(importances)]
print(f"1. {top_feat} dominates with {importances.max()*100:.1f}% MDI importance, reflecting the underlying mass-radius scaling law M ~ R^gamma.")
print(f"2. Insolation and semi-major axis contribute significantly to capturing envelope atmospheric retention vs. photoevaporation.")


### 6.4 Host-Star Grouped Cross-Validation (Fix 3)
Standard K-Fold may assign planets from the **same host star system** to both train and validation folds, creating an information leak (correlated stellar properties). We run **GroupKFold by hostname** to measure model performance when evaluated on unseen host star systems.


In [ ]:
# FIX 3: Host-star grouped cross-validation
# Groups = hostname; ensures no host star appears in both train and validation folds

groups_all = reg_df['hostname']  # hostname for all rows in reg_df
groups_train = groups_all.loc[X_train.index]  # align to X_train indices

gkf = GroupKFold(n_splits=5)
rf_pipe = fitted_pipelines['Random Forest']

group_cv_r2 = cross_val_score(rf_pipe, X_train, y_train,
                               cv=gkf, groups=groups_train, scoring='r2')
print(f"Random Forest Grouped CV R2 (by host star): {group_cv_r2.mean():.4f} +/- {group_cv_r2.std():.4f}")
print(f"Random Forest Standard KFold CV R2:          {results_table[results_table['Model']=='Random Forest']['CV R2 (Mean)'].values[0]:.4f}")
print()

diff = results_table[results_table['Model']=='Random Forest']['CV R2 (Mean)'].values[0] - group_cv_r2.mean()
if abs(diff) > 0.03:
    print(f"NOTE: Grouped CV R2 differs from standard KFold by {diff:.4f} — significant host-star correlation detected.")
else:
    print(f"NOTE: Grouped CV R2 is close to standard KFold (diff={diff:.4f}) — host-star correlation is minimal.")


---
# Section 7: Machine Learning — Demographic Clustering

### 7.1 Objective & Feature Space
In unsupervised clustering, our goal is to let the algorithm **discover exoplanet populations purely from data**, without using pre-defined labels.

### 7.2 Honest Clustering Assessment (FIX 4)

> **Scientific Note**: K-Means on the measured-mass subsample (`empirical_df`) was used in earlier analyses, but this subsample is dominated by gas giants (>70%). Small rocky planets rarely have measured dynamical masses, so the `log_pl_bmasse` feature is biased toward large planets. We therefore run clustering on **BOTH** subsamples:
>
> * **Subsample A**: `empirical_df` (measured mass; 4 features incl. log_pl_bmasse)
> * **Subsample B**: Full dataset `df` (3 features: log_pl_rade, log_pl_orbper, log_pl_insol — no mass requirement)

We note that **K-Means on the measured-mass subsample cannot recover a distinct rocky-planet cluster** because small planets with measured dynamical masses are severely under-represented in the archive relative to gas giants.


In [ ]:
# ── FIX 4: Honest Clustering ──────────────────────────────────────────────────

# --- Subsample A: empirical_df (mass measured; 4 features) ---
cluster_features_A = ['log_pl_rade', 'log_pl_bmasse', 'log_pl_orbper', 'log_pl_insol']
clust_A = empirical_df.dropna(subset=cluster_features_A).copy()
print(f"Subsample A (measured mass): {len(clust_A):,} planets")

scaler_A = StandardScaler()
X_A = scaler_A.fit_transform(clust_A[cluster_features_A])

sil_A = {}
for k in [2, 3, 4, 5]:
    km = KMeans(n_clusters=k, random_state=42, n_init=20)
    labels = km.fit_predict(X_A)
    sil_A[k] = silhouette_score(X_A, labels)
    print(f"  Subsample A k={k}: silhouette={sil_A[k]:.4f}  inertia={km.inertia_:.1f}")

# --- Subsample B: full df (no mass; 3 features) ---
cluster_features_B = ['log_pl_rade', 'log_pl_orbper', 'log_pl_insol']
clust_B = df.dropna(subset=cluster_features_B).copy()
print(f"\nSubsample B (full dataset, 3 features): {len(clust_B):,} planets")

scaler_B = StandardScaler()
X_B = scaler_B.fit_transform(clust_B[cluster_features_B])

sil_B = {}
for k in [2, 3, 4, 5]:
    km = KMeans(n_clusters=k, random_state=42, n_init=20)
    labels = km.fit_predict(X_B)
    sil_B[k] = silhouette_score(X_B, labels)
    print(f"  Subsample B k={k}: silhouette={sil_B[k]:.4f}  inertia={km.inertia_:.1f}")

print()
print("Silhouette summary:")
print(f"  Subsample A: {sil_A}")
print(f"  Subsample B: {sil_B}")


In [ ]:
# Pick best k for each subsample and label clusters HONESTLY

# --- Subsample A: best k ---
best_k_A = max(sil_A, key=sil_A.get)
print(f"Best k for Subsample A (measured-mass): k={best_k_A}, silhouette={sil_A[best_k_A]:.4f}")

km_A_final = KMeans(n_clusters=best_k_A, random_state=42, n_init=20)
clust_A['km_cluster'] = km_A_final.fit_predict(X_A)

profile_A = clust_A.groupby('km_cluster').agg(
    pl_rade_med=('pl_rade', 'median'),
    pl_bmasse_med=('pl_bmasse', 'median'),
    pl_orbper_med=('pl_orbper', 'median'),
    pl_insol_med=('pl_insol', 'median'),
    count=('pl_name', 'count')
).round(2)
print("\nSubsample A Cluster Profiles (Measured-Mass Subset):")
print(profile_A.to_string())

# Honest labeling based on ACTUAL median radii
def honest_label(r_med, p_med):
    if r_med < 2.0:
        return 'Small Planets (Terrestrial/Super-Earth regime)'
    elif r_med < 4.0:
        return 'Sub-Neptune/Neptune-like'
    elif r_med > 6.0 and p_med < 20:
        return 'Hot Jupiter'
    elif r_med > 6.0:
        return 'Jovian Gas Giant'
    else:
        return f'Mixed/Transitional (R={r_med:.1f} Rearth)'

label_map_A = {}
for cl in profile_A.index:
    r = profile_A.loc[cl, 'pl_rade_med']
    p = profile_A.loc[cl, 'pl_orbper_med']
    label_map_A[cl] = honest_label(r, p)

clust_A['cluster_label'] = clust_A['km_cluster'].map(label_map_A)
print("\nSubsample A Cluster Counts (honest labels):")
print(clust_A['cluster_label'].value_counts().to_string())

# --- Subsample B: best k ---
best_k_B = max(sil_B, key=sil_B.get)
print(f"\nBest k for Subsample B (full dataset): k={best_k_B}, silhouette={sil_B[best_k_B]:.4f}")

km_B_final = KMeans(n_clusters=best_k_B, random_state=42, n_init=20)
clust_B['km_cluster'] = km_B_final.fit_predict(X_B)

profile_B = clust_B.groupby('km_cluster').agg(
    pl_rade_med=('pl_rade', 'median'),
    pl_orbper_med=('pl_orbper', 'median'),
    pl_insol_med=('pl_insol', 'median'),
    count=('pl_name', 'count')
).round(2)
print("\nSubsample B Cluster Profiles (Full Dataset, 3 features):")
print(profile_B.to_string())

label_map_B = {}
for cl in profile_B.index:
    r = profile_B.loc[cl, 'pl_rade_med']
    p = profile_B.loc[cl, 'pl_orbper_med']
    label_map_B[cl] = honest_label(r, p)

clust_B['cluster_label'] = clust_B['km_cluster'].map(label_map_B)
print("\nSubsample B Cluster Counts (honest labels):")
print(clust_B['cluster_label'].value_counts().to_string())

print()
print("IMPORTANT CAVEAT (FIX 4):")
print("K-Means on the measured-mass subsample (Subsample A) cannot recover a distinct")
print("rocky-planet cluster because small planets with measured dynamical masses are")
print("severely under-represented in the archive relative to gas giants.")
print("The full-dataset clustering (Subsample B) provides a more representative picture")
print("of the actual planetary radius distribution.")


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

k_vals = [2, 3, 4, 5]

# Silhouette scores comparison
ax = axes[0]
ax.plot(k_vals, [sil_A[k] for k in k_vals], marker='o', linewidth=2.5, color='#e6550d', label='Subsample A (measured mass, 4 feat)')
ax.plot(k_vals, [sil_B[k] for k in k_vals], marker='s', linewidth=2.5, color='#2ca02c', label='Subsample B (full dataset, 3 feat)')
ax.axvline(best_k_A, color='#e6550d', linestyle='--', alpha=0.6, label=f'Best k_A={best_k_A}')
ax.axvline(best_k_B, color='#2ca02c', linestyle='--', alpha=0.6, label=f'Best k_B={best_k_B}')
ax.set_title('A. Silhouette Scores: Measured-Mass vs Full Dataset', fontsize=12, fontweight='bold')
ax.set_xlabel('Number of Clusters (k)', fontsize=11)
ax.set_ylabel('Mean Silhouette Score', fontsize=11)
ax.legend(frameon=True, fontsize=9)

# Scatter: Full-dataset clustering on Radius-Period plane
ax = axes[1]
unique_labels_B = clust_B['cluster_label'].unique()
palette_colors = ['#d62728', '#1f77b4', '#2ca02c', '#ff7f0e', '#9467bd']
palette_B = {lbl: palette_colors[i] for i, lbl in enumerate(sorted(unique_labels_B))}

for lbl, grp in clust_B.groupby('cluster_label'):
    if 'pl_orbper' in grp.columns and 'pl_rade' in grp.columns:
        ax.scatter(grp['pl_orbper'], grp['pl_rade'], s=12, alpha=0.5,
                   label=f"{lbl} (n={len(grp):,})", color=palette_B.get(lbl, 'gray'))

ax.set_xscale('log')
ax.set_yscale('log')
ax.set_title(f'B. Full-Dataset Clustering (k={best_k_B}) — Period-Radius Plane', fontsize=12, fontweight='bold')
ax.set_xlabel('Orbital Period [days] (log scale)', fontsize=11)
ax.set_ylabel('Planet Radius [Rearth] (log scale)', fontsize=11)
ax.legend(frameon=True, fontsize=8, loc='upper left')

plt.tight_layout()
plt.savefig('figures/07_clustering_honest_assessment.png', dpi=300)
plt.show()


## Summary of Findings (Sections 6 & 7)

### Machine Learning Prediction (Pipeline-corrected):
1. **Model Performance**: Non-linear ensemble methods significantly outperform linear models. Results reported with **leakage-free sklearn Pipelines** that fit StandardScaler only on training data within each CV fold.
2. **Log-scale Error Interpretation**: Errors are in log10(M_earth) units (dex). An RMSE of ~0.35 dex corresponds to a factor of 10^0.35 ≈ 2.24 multiplicative uncertainty in predicted mass. A MedAE of ~0.11 dex corresponds to a typical median multiplicative error of 10^0.11 ≈ 1.29 (29% typical relative uncertainty in mass).
3. **Primary Predictive Drivers**: Planet radius (MDI: Mean Decrease in Impurity via MSE Reduction — *not* Gini, which applies only to classification) dominates, followed by stellar insolation and semi-major axis.
4. **Mass Provenance**: The `is_measured_mass` flag includes both true dynamical masses and Msini (minimum masses from RV), with mean statistical correction ⟨sin i⟩ = π/4 ≈ 0.785 for inclination.
5. **Grouped CV**: Host-star grouped cross-validation (GroupKFold) tests generalization to unseen planetary systems, avoiding cross-contamination of correlated stellar features.

### Machine Learning Clustering (Honest Assessment):
1. **Measured-Mass Subsample Limitation**: K-Means on the measured-mass subsample cannot recover a distinct rocky-planet cluster. This subsample is dominated (>70%) by gas giants; small rocky planets rarely have measured dynamical masses in the archive.
2. **Full-Dataset Clustering**: Running K-Means on the full dataset using 3 mass-agnostic features (radius, period, insolation) gives a more representative view of the exoplanet radius-period demographics.
3. **Cluster labels are assigned honestly** based on actual median radii and periods — not assumed prior categories.
